In [20]:
import sys
import os
sys.path.append('../')

import pandas as pd
from data_clients.polymarket_client import PolymarketAPIClient
from util.data_processor import parse_timestamp, TickDataIntervalEnum
from trading_rules.position_data import Positions
from trading_rules.mean_reversal import MeanReversal
from trading_rules.market_data import MarketData
from util.porfolio_performance import PortfolioPerformance
from util.backtester import perform_mean_reversal_backtest

In [21]:
import vectorbt as vbt

In [ ]:
def perform_mean_reversal(market_df):
    # Mean Reversion strategy using vectorbt

    import itertools
    
    
    price = pd.Series(market_df['close'])  # your close prices
    price = price.ffill()
    
    X_vals = [5, 10, 20]   # lookback window in days
    Y_vals = [1, 2, 3, 5]  # holding period in days
    
    STARTING_CASH = 10000
    FEES = 0.001
    
    # 10-min bars per day
    BARS_PER_DAY = 144  # 6 bars/hour * 24 hours
    
    results = []
    
    for X, Y in itertools.product(X_vals, Y_vals):
        window = X * BARS_PER_DAY   # convert days → bars
        hold   = Y * BARS_PER_DAY   # convert days → bars
    
        # Rolling minimum
        rolling_min = price.rolling(window).min()
    
        # Entry: price hits or falls below the X-day rolling min
        entries = price <= rolling_min
    
        # Exit: after exactly Y days (using td param in from_signals)
        # vectorbt supports sl_stop, tp_stop, time_delta_ns etc.
        # Cleanest approach: build exit signals manually via shift
        exits = entries.shift(hold).fillna(False).astype(bool)
    
        pf = vbt.Portfolio.from_signals(
            close=price,
            entries=entries,
            exits=exits,
            fees=FEES,
            freq="10min",
            init_cash=1000,
        )
    
        stats = pf.stats()
    
        results.append({
            "X(d)": X,
            "Y(d)": Y,
            "Start Value": round(stats["Start Value"]),
            "End Value": round(stats["End Value"]),
            # "CAR": round(stats["Ann. Return [%]"], 2),
            # "Ann. Vol": round(stats["Ann. Volatility [%]"], 2),
            "Total Return": round(stats["Total Return [%]"]),
            "Sharpe": round(stats["Sharpe Ratio"], 2),
            "Max DD": round(stats["Max Drawdown [%]"], 2),
            "Max Drawdown [%]": round(stats["Max Drawdown [%]"]),
            "Win Rate [%]": round(stats["Win Rate [%]"]),
            "Total Trades": round(stats["Total Trades"])
    
            # "Calmar": round(stats["Ann. Return [%]"] / abs(stats["Max Drawdown [%]"]), 2)
            #     if stats["Max Drawdown [%]"] != 0 else np.nan,
        })
    return results
    

In [23]:
MARKET_SLUG = 'will-jesus-christ-return-in-2025'

client = PolymarketAPIClient()
market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
market_data = MarketData(market)
positions = Positions(cash=10000.0)

resampled_df = market.resample("10min").last()
market_data = MarketData(resampled_df)

Will Jesus Christ return in 2025?
Not Will Jesus Christ return in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


In [24]:
# Mean Reversion strategy using vectorbt
results_jesus = perform_mean_reversal(market_data.df)
results_jesus_df = pd.DataFrame(results_jesus)


C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(cop

In [25]:
MARKET_SLUG = "will-china-invade-taiwan-in-2025"

market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
market_data = MarketData(market)
positions = Positions(cash=10000.0)

resampled_df = market.resample("10min").last()
market_data = MarketData(resampled_df)

Will China invade Taiwan in 2025?
Not Will China invade Taiwan in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


In [26]:
results_china = perform_mean_reversal(market_data.df)
results_china_df = pd.DataFrame(results_china)

C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(cop

In [27]:
MARKET_SLUG = "will-the-us-confirm-that-aliens-exist-in-2025"

market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
market_data = MarketData(market)
positions = Positions(cash=10000.0)

resampled_df = market.resample("10min").last()
market_data = MarketData(resampled_df)

Will the US confirm that aliens exist in 2025?
Not Will the US confirm that aliens exist in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


In [28]:
results_alien = perform_mean_reversal(market_data.df)
results_alien_df = pd.DataFrame(results_alien)

C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  exits = entries.shift(hold).fillna(False).astype(bool)
C:\Users\manug\AppData\Local\Temp\ipykernel_15168\3005547644.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(cop

In [29]:
# Sort the three dataframes by the higest sharpe ratios
col = "Sharpe"
results_jesus_df.sort_values(by=col, ascending=False, inplace=True)
results_alien_df.sort_values(by=col, ascending=False, inplace=True)
results_china_df.sort_values(by=col, ascending=False, inplace=True)

In [30]:
results_jesus_df

,X(d),Y(d),Start Value,End Value,Total Return,Sharpe,Max DD,Max Drawdown [%],Win Rate [%],Total Trades
11,20,5,1000,1015,2,0.67,0.71,1,78,9
9,20,2,1000,1010,1,0.52,1.04,1,45,11
7,10,5,1000,1007,1,0.28,1.23,1,67,18
10,20,3,1000,1005,1,0.25,1.17,1,38,8
8,20,1,1000,1003,0,0.18,0.96,1,54,13
3,5,5,1000,998,0,-0.06,1.86,2,53,32
6,10,3,1000,992,-1,-0.31,2.32,2,32,22
5,10,2,1000,991,-1,-0.39,2.65,3,31,26
2,5,3,1000,988,-1,-0.42,2.64,3,38,34
1,5,2,1000,982,-2,-0.69,3.64,4,33,43


In [31]:
results_alien_df

,X(d),Y(d),Start Value,End Value,Total Return,Sharpe,Max DD,Max Drawdown [%],Win Rate [%],Total Trades
7,10,5,1000,1876,88,1.67,11.14,11,84,20
11,20,5,1000,1815,81,1.67,10.91,11,100,12
10,20,3,1000,1736,74,1.56,10.91,11,83,12
9,20,2,1000,1719,72,1.54,10.91,11,92,12
8,20,1,1000,1705,70,1.52,10.91,11,92,12
6,10,3,1000,1731,73,1.49,11.14,11,76,21
5,10,2,1000,1567,57,1.27,14.22,14,71,21
1,5,2,1000,1546,55,1.22,15.88,16,67,43
0,5,1,1000,1528,53,1.20,15.88,16,65,51
4,10,1,1000,1519,52,1.20,14.22,14,62,24


In [32]:
results_china_df

,X(d),Y(d),Start Value,End Value,Total Return,Sharpe,Max DD,Max Drawdown [%],Win Rate [%],Total Trades
4,10,1,1000,2888,189,1.96,4.70,5,81,31
7,10,5,1000,2905,191,1.96,4.32,4,83,19
6,10,3,1000,2842,184,1.93,4.35,4,85,26
5,10,2,1000,2823,182,1.92,4.70,5,77,26
11,20,5,1000,2806,181,1.92,3.11,3,78,10
8,20,1,1000,2743,174,1.89,4.90,5,80,15
9,20,2,1000,2712,171,1.87,4.90,5,77,13
10,20,3,1000,2690,169,1.85,4.99,5,75,12
0,5,1,1000,2353,135,1.59,26.31,26,75,52
2,5,3,1000,2264,126,1.53,25.95,26,70,37
